In [ ]:
!pip install -q -U transformers accelerate datasets huggingface_hub matplotlib
!pip install -q sae-lens

In [ ]:
# import sys
# sys.path.insert(1, "/kaggle/input/corpus_loader")

import os
import time
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sae_lens import SAE

from corpus_loader import CorpusConfig, chunk_stream, batch_chunks

In [ ]:
# ---- модель ----
MODEL_ID = "google/gemma-2-2b"
LAYER_IDX = 12                      
DTYPE = torch.float16             

# ---- SAE (две ширины, один и тот же слой) ----
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_IDS = {
    "16k": f"layer_{LAYER_IDX}/width_16k/canonical",
    "65k": f"layer_{LAYER_IDX}/width_65k/canonical",
}

# ---- корпус ----
SEQ_LEN = 1024
TARGET_TOKENS = 30_000_000       
BATCH_SIZE = 8                      

# ---- гистограмма ----
N_BINS = 64
BIN_MIN = 1e-3
BIN_MAX = 100.0                    

# ---- инфраструктура ----
SECRET_NAME = "llama-token"        
OUTPUT_DIR = "/kaggle/working"
CKPT_PATH = os.path.join(OUTPUT_DIR, "pass1_gemma_checkpoint.npz")
CHECKPOINT_EVERY_N_BATCHES = 200
MAX_RUNTIME_HOURS = 8.5             

In [ ]:
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret(SECRET_NAME)
    print("secret ok:", hf_token[:7] + "...")
except Exception as e:
    print("Секрет не получен:", repr(e))
    print("Задай hf_token вручную, если нужно.")

if hf_token:
    os.environ["HF_TOKEN"] = hf_token

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto",
)
model.eval()

device = next(model.parameters()).device
print("device:", device, "| dtype:", next(model.parameters()).dtype)
print("n_layers:", len(model.model.layers), "| d_model:", model.config.hidden_size)

In [ ]:
saes = {}
for name, sae_id in SAE_IDS.items():
    out = SAE.from_pretrained(release=SAE_RELEASE, sae_id=sae_id, device=str(device))
    sae = out[0] if isinstance(out, tuple) else out
    sae.eval()
    saes[name] = sae
    print(f"{name}: W_dec.shape = {tuple(sae.W_dec.shape)}")

D = {}
for name, sae in saes.items():
    a, b = sae.W_dec.shape
    assert b == model.config.hidden_size, (
        f"{name}: W_dec.shape={tuple(sae.W_dec.shape)}, "
    )
    D[name] = a
    print(f"{name}: D = {a} фич")

In [ ]:
with torch.no_grad():
    dummy = torch.randn(4, model.config.hidden_size, device=device, dtype=saes["16k"].W_enc.dtype)
    z = saes["16k"].encode(dummy)
    print("encode() -> тип:", type(z), "| форма:", tuple(z.shape))
    print("ненулевых на строку (L0):", (z != 0).sum(dim=1).tolist())
    print("макс активация:", z.max().item())

In [ ]:
_captured = {}

def _hook_fn(module, inputs, output):
    # Блок Gemma возвращает кортеж, hidden state — первый элемент
    _captured["hidden"] = output[0] if isinstance(output, tuple) else output

target_module = model.model.layers[LAYER_IDX]
handle = target_module.register_forward_hook(_hook_fn)
print(f"Hook на model.model.layers[{LAYER_IDX}] (residual stream, post-block)")

In [ ]:
def make_bin_edges(n_bins, vmin, vmax):
    inner = np.logspace(np.log10(vmin), np.log10(vmax), n_bins - 1)
    return np.concatenate([[0.0], inner, [np.inf]]).astype(np.float64)

BIN_EDGES = make_bin_edges(N_BINS, BIN_MIN, BIN_MAX)


def save_checkpoint(path, hists, tokens_seen, chunks_done):
    tmp = path + ".tmp"
    payload = {f"hist_{k}": v for k, v in hists.items()}
    payload["bin_edges"] = BIN_EDGES
    payload["tokens_seen"] = np.int64(tokens_seen)
    payload["chunks_done"] = np.int64(chunks_done)
    with open(tmp, "wb") as f:
        np.savez_compressed(f, **payload)
    os.replace(tmp, path)


def load_checkpoint(path, widths):
    if not os.path.exists(path):
        return None
    data = np.load(path)
    return {
        "hists": {w: data[f"hist_{w}"] for w in widths},
        "tokens_seen": int(data["tokens_seen"]),
        "chunks_done": int(data["chunks_done"]),
    }


ckpt = load_checkpoint(CKPT_PATH, list(SAE_IDS.keys()))
if ckpt is not None:
    hists = ckpt["hists"]
    tokens_seen = ckpt["tokens_seen"]
    chunks_done = ckpt["chunks_done"]
    print(f"Резюмируем: tokens_seen={tokens_seen:,}, chunks_done={chunks_done:,}")
else:
    hists = {name: np.zeros((D[name], N_BINS), dtype=np.int64) for name in SAE_IDS}
    tokens_seen = 0
    chunks_done = 0
    print("Чекпоинт не найден, начинаем с нуля")

for name, h in hists.items():
    print(f"  {name}: аккумулятор {h.shape}, {h.nbytes / 1e6:.1f} MB")

skip_tokens = chunks_done * SEQ_LEN

In [ ]:
def accumulate(hist_acc, feat_idx, values, bin_edges):
    '''Обновляет гистограмму по ненулевым активациям.
    feat_idx, values — плоские массивы одинаковой длины.'''
    if feat_idx.size == 0:
        return
    bin_idx = np.searchsorted(bin_edges, values, side="right") - 1
    np.clip(bin_idx, 0, len(bin_edges) - 2, out=bin_idx)
    flat = feat_idx.astype(np.int64) * hist_acc.shape[1] + bin_idx
    hist_acc += np.bincount(flat, minlength=hist_acc.size).reshape(hist_acc.shape)

In [ ]:
cfg = CorpusConfig(seq_len=SEQ_LEN, skip_tokens=skip_tokens, max_tokens=TARGET_TOKENS)
batches = batch_chunks(chunk_stream(tokenizer, cfg), BATCH_SIZE)

start = time.time()
n_batches = 0

with torch.no_grad():
    for batch in batches:
        elapsed_h = (time.time() - start) / 3600
        if elapsed_h > MAX_RUNTIME_HOURS:
            print(f"Бюджет времени ({MAX_RUNTIME_HOURS}ч) исчерпан, сохраняемся.")
            break

        batch = batch.to(device)
        model(batch)                                   

        hidden = _captured["hidden"]                  
        hidden_flat = hidden.reshape(-1, hidden.shape[-1])

        for name, sae in saes.items():
            z = sae.encode(hidden_flat.to(sae.W_enc.dtype))   
            nz = z.nonzero(as_tuple=True)                     
            feat_idx = nz[1].cpu().numpy()
            values = z[nz].float().cpu().numpy()
            accumulate(hists[name], feat_idx, values, BIN_EDGES)
            del z, nz

        tokens_seen += batch.shape[0] * batch.shape[1]
        chunks_done += batch.shape[0]
        n_batches += 1

        if n_batches % CHECKPOINT_EVERY_N_BATCHES == 0:
            save_checkpoint(CKPT_PATH, hists, tokens_seen, chunks_done)
            rate = tokens_seen / max(time.time() - start, 1)
            print(f"[ckpt] tokens={tokens_seen:,} | {elapsed_h:.2f}ч | ~{rate:.0f} tok/s")

save_checkpoint(CKPT_PATH, hists, tokens_seen, chunks_done)
print(f"Готово. tokens_seen={tokens_seen:,}, chunks_done={chunks_done:,}")

handle.remove()

## Sanity-check

Не анализ (GMM/dip test — отдельный шаг локально), а проверка что числа вменяемые.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(hists) + 1, figsize=(5 * (len(hists) + 1), 4))

for ax, (name, h) in zip(axes, hists.items()):
    density = h.sum(axis=1) / max(tokens_seen, 1)
    alive = density > 0
    ax.hist(np.log10(density[alive]), bins=50)
    ax.set_title(f"{name}: log10(density)\n{alive.sum()}/{D[name]} живых")
    ax.set_xlabel("log10(доля ненулевых токенов)")

h16 = hists["16k"]
top = int(np.argmax(h16.sum(axis=1)))
axes[-1].bar(range(N_BINS), h16[top])
axes[-1].set_title(f"16k: фича #{top} (самая частая)")
axes[-1].set_xlabel("bin index (лог-шкала)")

plt.tight_layout()
plt.show()

for name, h in hists.items():
    counts = h.sum(axis=1)
    print(f"{name}: фич с >=300 срабатываний: {(counts >= 300).sum()} / {D[name]}")
    top_bin_share = h[:, -1].sum() / max(h.sum(), 1)
    print(f"  доля попаданий в overflow-бин: {top_bin_share:.4%}"
          f"{'  <-- подними BIN_MAX!' if top_bin_share > 0.01 else ''}")